# 이미지 분류 1일차 실습 1 — 확률 · 흔들기 · 혼동 행렬 · 세 칸

확률을 직접 만든다 → 한 종류만 흔들어 본다 → 표를 가로 · 세로로 읽는다 → 한 짝을 세 칸으로 적는다

- 이 노트북 · `practice_data.npz` · `d1_base_best.pt` 세 파일이 같은 폴더에 있어야 함 (Colab: 왼쪽 파일 창에 세 파일을 올림)
- 모델은 수업에서 쓴 저장 모델(증강 없음) · 사진은 검증 사진 앞 2000장 — 사진 수가 달라 수업 화면 숫자와 조금 다름
- ◆ 표시가 있는 줄이 고칠 줄 · 고치기 전에도 돌아감 · 칸 끝의 **확인** 줄이 제대로 고쳤는지 알려 줌
- 문제마다 순서: **예상 먼저**(노트나 속으로) → ◆ 줄 고치기 → 실행 → 확인 줄 읽기
- 막히면 문제 아래 "막혔을 때 이어갈 코드" 칸을 실행하고 다음 문제로 넘어갈 것
- 난이도 — ★ 기본(모두 · 한두 줄 고치기) · ★★ 도전(조건 하나를 스스로 바꿔 비교) · ★★★ 심화(짧은 코드 몇 줄 작성) · 수업 시간은 ★ 기준 · ★★ · ★★★ 는 먼저 끝난 사람이 이어서

## 0. 준비 — 한 번만 실행

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.v2.functional as TF
import matplotlib.pyplot as plt

data = np.load("practice_data.npz")
x_val, y_val = torch.from_numpy(data["x_val"]), torch.from_numpy(data["y_val"])
CLASSES = [str(c) for c in data["classes"]]


class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        layers, c = [], 3
        for w in (32, 64, 128):
            layers += [nn.Conv2d(c, w, 3, padding=1), nn.BatchNorm2d(w), nn.ReLU(),
                       nn.Conv2d(w, w, 3, padding=1), nn.BatchNorm2d(w), nn.ReLU(), nn.MaxPool2d(2)]
            c = w
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(c, n_classes))

    def forward(self, x):
        return self.head(self.features(x))


def to_input(x):
    return (x.float() / 255 - 0.5) / 0.25


@torch.no_grad()
def predict(model, x, bs=500):
    model.eval()
    return torch.cat([model(to_input(x[k:k + bs])).softmax(1) for k in range(0, len(x), bs)])


def show(images, titles, cols=8, size=1.6):
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * size, rows * (size + 0.4)), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, im, t in zip(axes.flat, images, titles):
        ax.imshow((im.float().clamp(0, 255) / 255).permute(1, 2, 0).numpy(), interpolation="nearest")
        ax.set_title(t, fontsize=8)
    fig.tight_layout()
    plt.show()


model = SmallCNN(len(CLASSES))
model.load_state_dict(torch.load("d1_base_best.pt", map_location="cpu"))
p_val = predict(model, x_val)                         # 사진마다 열 가지 확률 (2000, 10)
conf, pred = p_val.max(1)                             # 가장 큰 확률 = 확신 · 그 종류 = 모델의 답
correct = (pred == y_val).nonzero().flatten()
print(f"준비 끝 · 검증 사진 {len(x_val)}장 · 모델이 맞힌 사진 {len(correct)}장")

**에러가 나면 첫 줄부터 읽을 것**
- `FileNotFoundError` — 세 파일이 노트북과 같은 폴더에 없음 · Colab 파일 창에 올렸는지 확인
- `NameError: name 'model' is not defined` — 이 준비 칸을 건너뜀 · 맨 위부터 다시 실행

## 개념 문제 — 객관식 여섯 개 (약 4분)

- 보기 네 개 중 하나를 골라 아래 코드 칸의 `answers` 에 글자(A · B · C · D)로 적을 것 · 채점은 하지 않음 · 해설 시간에 함께 봄
- 코드는 실행하지 않고 생각만으로 고를 것

**Q1. 점수(logit)가 `[2.0, 1.0, -1.0]` 인 사진에 softmax 를 쓰면?**
- A. 음수 점수 -1.0 인 종류의 확률은 0 이 됨
- B. 세 확률의 합은 점수의 합인 2.0 이 됨
- C. 확률의 크기 순서는 점수의 크기 순서와 같음
- D. 점수 차이가 작으면 순서가 뒤바뀔 수 있음

**Q2. 모델이 어떤 사진에 가장 큰 확률 0.97 을 주었는데 틀렸음. 가장 알맞은 해석은?**
- A. 확률이 0.97 이면 틀릴 수 없으니 정답 표시가 잘못됨
- B. 확률은 모델의 확신일 뿐 · 자신 있게 틀린 사진도 있음
- C. 확률이 높을수록 손실은 커짐
- D. softmax 를 두 번 써야 제대로 된 확률이 나옴

**Q3. 흔들기에서 '답이 바뀐 비율'의 분모로 알맞은 것은?**
- A. 검증 사진 전체
- B. 모델이 원본을 맞힌 사진
- C. 모델이 원본을 틀린 사진
- D. 학습 사진 전체

**Q4. 학습 곡선에서 과적합(외움)을 보여 주는 모양은?**
- A. 학습 정확도가 높음 — 그것만으로 과적합
- B. 학습 선 · 검증 선 모두 낮고 둘 다 아직 오르는 중
- C. 학습 선은 계속 오르는데 검증 선은 멈추거나 내려가며 간격이 벌어짐
- D. 검증 손실이 계속 내려감

**Q5. checkpoint 로 '가장 좋았던 모델'을 고를 때 기준으로 쓰는 사진은?**
- A. 학습 사진 — 가장 많이 봤으니 가장 정확함
- B. 검증 사진
- C. 테스트 사진 — 마지막 성적이니 처음부터 기준으로
- D. 기준 없이 마지막 에폭 모델

**Q6. 혼동 행렬(줄 = 정답 · 열 = 모델의 답)에서 '모델이 트럭이라고 답하면 얼마나 믿을 수 있나'를 재려면?**
- A. 트럭 줄의 대각선 칸 ÷ 트럭 줄의 합 (가로로 읽기)
- B. 트럭 열의 대각선 칸 ÷ 트럭 열의 합 (세로로 읽기)
- C. 트럭 대각선 칸 ÷ 표 전체의 합
- D. 트럭 줄의 합 ÷ 트럭 열의 합

In [ ]:
answers = {"Q1": "", "Q2": "", "Q3": "", "Q4": "", "Q5": "", "Q6": ""}                                       # ◆ 따옴표 안에 A · B · C · D 중 하나

filled = [q for q, a in answers.items() if a.strip().upper() in ("A", "B", "C", "D")]
print(f"적은 답 {len(filled)}개 / 6개 —", answers)

## 문제 1 ★ 기본 — 점수를 확률로 직접 바꾸기 (softmax · top-k)

- **왜** — 수업에서 `softmax` 한 줄로 점수를 확률로 바꿈 · 그 한 줄 안에서 무슨 일이 일어나는지 손으로 해 봄
- **목표** — 가장 자신 있게 틀린 사진 한 장의 점수 열 개를 확률 열 개로 바꾸고, 큰 순서 k개를 찍기
- **예상 먼저** — 점수가 가장 큰 종류와 확률이 가장 큰 종류는 같을까 다를까
- **◆ 고칠 줄** — `prob = logit.exp()` 뒤에 **합으로 나누기**를 채울 것 (두 번째 ◆ 는 k)
- 문법 예시
  - 하나하나 지수 함수: `logit.exp()` — 음수 점수도 양수가 됨
  - 모두 더하기: `a.sum()` · 나누기: `a / a.sum()` — 열 개를 더하면 1
  - 큰 순서 k개: `vals, inds = prob.topk(k)` — 값과 종류 번호를 함께 돌려줌

In [ ]:
wrong = (pred != y_val).nonzero().flatten()
order = wrong[conf[wrong].argsort(descending=True)]  # 틀린 사진을 확신이 큰 순서로
show([x_val[i] for i in order[:8]],
     [f"[{r}] true {CLASSES[y_val[i]]}\npred {CLASSES[pred[i]]} {conf[i]:.2f}" for r, i in enumerate(order[:8])])

pick = 0                                              # 위 그림의 [번호] — 0 은 가장 자신 있게 틀린 사진
i = int(order[pick])
model.eval()
with torch.no_grad():
    logit = model(to_input(x_val[i:i + 1]))[0]       # 점수 열 개

prob = logit.exp()                       # ◆ 합으로 나누기를 채울 것 — 예: a / a.sum()
k = 1                                                 # ◆ 큰 순서로 몇 개 볼지 — 1 ~ 10

vals, inds = prob.topk(k)
print(f"{'순서':4s} {'종류':12s} {'점수':>8s} {'확률':>7s}")
for r, (v, c) in enumerate(zip(vals, inds), start=1):
    mark = "  ← 정답" if int(c) == int(y_val[i]) else ""
    print(f"{r:4d} {CLASSES[c]:12s} {logit[c]:8.2f} {v:7.3f}{mark}")
rank_true = int((prob > prob[y_val[i]]).sum()) + 1
print(f"정답 {CLASSES[y_val[i]]} 은 확률 순서로 {rank_true}째")

# 확인 — 고치지 않았으면 무엇이 덜 되었는지 알려 줌
same = torch.allclose(prob, logit.softmax(0), atol=1e-5)
print(f"확인 · 확률 합 {prob.sum():.3f} (1이어야 함) · softmax 한 줄과 같은가 {same}")
if not same:
    print("   → 아직 합으로 나누지 않음 · ◆ 줄을 a / a.sum() 모양으로 고칠 것")
print(f"확인 · 점수 1등 {CLASSES[logit.argmax()]} · 확률 1등 {CLASSES[prob.argmax()]}")

- 확인 질문 1 — 점수 1등과 확률 1등이 같은가 · 왜 그런가
- 확인 질문 2 — k 를 3 으로 바꾸면 정답이 보이는가 · 보인다면 몇째인가
- 흔한 에러
  - `RuntimeError: selected index k out of range` — `k` 가 10보다 큼 · 종류는 열 개뿐
  - `IndexError` — `pick` 이 틀린 사진 장수보다 큼
  - `logit.exp() / sum` 처럼 괄호를 빠뜨리면 `TypeError` — 나눌 것은 `logit.exp().sum()` 전체

**막혔을 때 이어갈 코드** — 가장 자신 있게 틀린 사진의 큰 순서 세 개

In [ ]:
wrong = (pred != y_val).nonzero().flatten()
i = int(wrong[conf[wrong].argmax()])
vals, inds = p_val[i].topk(3)
print(f"정답 {CLASSES[y_val[i]]} · 큰 순서 세 개 —", " · ".join(f"{CLASSES[c]} {v:.3f}" for v, c in zip(vals, inds)))

### 문제 1 ★★ 도전 — 확신 기준을 바꿔 맞힌 사진 · 틀린 사진 견주기

- 조건 하나(`threshold`)를 바꿔, 확신이 기준 이상인 사진이 맞힌 쪽 · 틀린 쪽에 각각 몇 장인지 견줌
- 예상 먼저 — 기준을 0.99 로 올리면 틀린 사진 중 남는 비율과 맞힌 사진 중 남는 비율 중 어느 쪽이 더 많이 줄어들까

In [ ]:
threshold = 0.9                                      # ◆ 0.5 · 0.9 · 0.99 로 바꿔 가며 실행
right, wrong_ = (pred == y_val), (pred != y_val)
hi = conf >= threshold
print(f"확신 {threshold} 이상 — 맞힌 사진 {int((hi & right).sum())}/{int(right.sum())}장 ({(hi & right).sum() / right.sum() * 100:.1f}%)"
      f" · 틀린 사진 {int((hi & wrong_).sum())}/{int(wrong_.sum())}장 ({(hi & wrong_).sum() / wrong_.sum() * 100:.1f}%)")

### 문제 1 ★★★ 심화 — top-3 정확도 직접 계산 (몇 줄 작성)

- 사진마다 확률 큰 순서 세 종류 안에 정답이 있으면 맞힌 것으로 셈
- 문법 예시: `p_val.topk(3, dim=1).indices` → (2000, 3) · `(a == y_val[:, None]).any(1)` → 줄마다 하나라도 같은가

In [ ]:
top1 = (pred == y_val).float().mean().item()
hit = None                                            # ★★★ 여기에 두세 줄 작성 — 마지막에 hit = 0~1 사이 값

if hit is None:
    print("아직 작성 안 함 — 문법 예시 두 줄을 이어 붙일 것")
else:
    print(f"top-1 정확도 {top1 * 100:.1f}% · top-3 정확도 {hit * 100:.1f}%")

## 문제 2 ★ 기본 — 한 종류만 골라 흔들기

- **왜** — 수업에서는 맞힌 사진 **전체**를 여섯 가지로 흔듦 · 종류마다 약한 변화가 다른지 봄
- **목표** — 종류 하나를 골라 여섯 변화에서 '답이 바뀐 비율'을 전체와 나란히 견주기
- **예상 먼저** — 고른 종류가 전체보다 더 잘 흔들릴 변화 하나를 먼저 적을 것
- **◆ 고칠 줄** — `kind` 한 줄 (두 번째 칸은 `which`)
- 분모 — 모델이 원본을 맞힌 사진 (전체 · 고른 종류 각각) · 여섯 변화와 세기는 수업과 같음
- 문법 예시: 답이 바뀐 비율 `(새 답 != 원래 답).float().mean()` — 참은 1, 거짓은 0 으로 바꿔 평균
- 종류 이름: airplane · automobile · bird · cat · deer · dog · frog · horse · ship · truck

In [ ]:
KINDS = ["hflip", "shift", "dark", "bright", "crop", "rotate"]


def shake(x, kind):
    """사진 묶음(N, 3, 32, 32)에 변화 하나를 줌. 정답은 그대로 둠. 수업과 같은 세기."""
    x = x.float()
    if kind == "hflip":
        return x.flip(-1)
    if kind == "shift":
        return TF.affine(x, angle=0.0, translate=[4, 4], scale=1.0, shear=[0.0, 0.0])
    if kind == "dark":
        return (x * 0.55).clamp(0, 255)
    if kind == "bright":
        return (x * 1.5).clamp(0, 255)
    if kind == "crop":
        return TF.resize(TF.center_crop(x, [22, 22]), [32, 32], antialias=True)
    if kind == "rotate":
        return TF.rotate(x, 20.0)
    raise ValueError(kind)


kind = "cat"                                          # ◆ 흔들어 볼 종류

mine = correct[y_val[correct] == CLASSES.index(kind)] # 원본을 맞힌 사진 중 고른 종류
rate_all, rate_mine = [], []
for ch in KINDS:
    new_all = predict(model, shake(x_val[correct], ch)).argmax(1)
    rate_all.append((new_all != pred[correct]).float().mean().item() * 100)
    new_mine = predict(model, shake(x_val[mine], ch)).argmax(1)
    rate_mine.append((new_mine != pred[mine]).float().mean().item() * 100)

y_ = np.arange(len(KINDS))
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.barh(y_ + 0.2, rate_all, 0.4, color="#7aa2f7", label="all correct")
ax.barh(y_ - 0.2, rate_mine, 0.4, color="#f7768e", label=f"{kind} only")
ax.set_yticks(y_, KINDS)
ax.invert_yaxis()
ax.set_xlabel("answer changed (%)")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()
print(f"맞힌 사진 전체 {len(correct)}장 · 그중 {kind} {len(mine)}장")
for ch, a, m in zip(KINDS, rate_all, rate_mine):
    print(f"{ch:7s} 전체 {a:5.1f}% · {kind} {m:5.1f}%  {'← 더 흔들림' if m > a else ''}")

In [ ]:
which = "crop"                                        # ◆ 사진으로 볼 변화 — hflip · shift · dark · bright · crop · rotate

new = predict(model, shake(x_val[mine], which)).argmax(1)
moved = (new != pred[mine]).nonzero().flatten()[:8]
if len(moved) == 0:
    print(f"{which} 에서는 {kind} 사진의 답이 하나도 바뀌지 않음 — 다른 변화를 고를 것")
else:
    tops = [x_val[mine[j]] for j in moved]
    bottoms = [shake(x_val[mine[j]][None], which)[0] for j in moved]
    show(tops + bottoms,
         [f"true {kind}" for _ in moved] + [f"{which}\n-> {CLASSES[new[j]]}" for j in moved], cols=len(moved))
    moved_all = (new != pred[mine]).nonzero().flatten()
    top_new = torch.bincount(new[moved_all], minlength=len(CLASSES)).argmax()
    print(f"{which} 로 답이 바뀐 {kind} 사진 {len(moved_all)}장 · 위 원본 / 아래 바뀐 사진과 새 답")
    print(f"새 답으로 가장 많이 몰린 종류 {CLASSES[top_new]}")

- 확인 질문 1 — 예상한 변화에서 빨간 막대가 파란 막대보다 긴가 · 출력 줄의 `← 더 흔들림` 표시로 확인
- 확인 질문 2 — 새 답이 어느 종류로 몰리는가 · 바뀐 사진은 사람 눈에도 달라 보이는가
- 주의 — 한 종류는 장수가 적어 몇 장 차이로도 비율이 크게 움직임
- 흔한 에러
  - `ValueError: 'Cat' is not in list` — 종류 이름은 소문자 영어 · 목록 철자 그대로
  - `ValueError: crops` — 변화 이름 철자가 목록과 다름

**막혔을 때 이어갈 코드** — 고양이만 좌우 반전

In [ ]:
mine = correct[y_val[correct] == CLASSES.index("cat")]
new = predict(model, x_val[mine].float().flip(-1)).argmax(1)
print(f"고양이 {len(mine)}장 중 좌우 반전으로 답이 바뀐 비율 {(new != pred[mine]).float().mean().item() * 100:.1f}%")

### 문제 2 ★★ 도전 — 회전 세기를 바꿔 고른 종류와 전체 견주기

- 조건 하나(회전 각도 목록)를 바꿔, 세기가 커질 때 두 비율이 어떻게 벌어지는지 봄

In [ ]:
angles = [20]                                      # ◆ 각도 목록 — 예: [5, 20, 45]
for a in angles:
    r_all = (predict(model, TF.rotate(x_val[correct].float(), float(a))).argmax(1) != pred[correct]).float().mean() * 100
    r_mine = (predict(model, TF.rotate(x_val[mine].float(), float(a))).argmax(1) != pred[mine]).float().mean() * 100
    print(f"회전 {a:3d}도 · 전체 {r_all:5.1f}% · {kind} {r_mine:5.1f}%")

### 문제 2 ★★★ 심화 — 한 변화에 가장 약한 종류 찾기 (몇 줄 작성)

- `shake` 와 `which` 를 그대로 쓰고, 열 종류마다 '답이 바뀐 비율'을 구해 `rates` 목록에 담을 것
- 문법 예시: `for c in range(len(CLASSES)):` · `m = correct[y_val[correct] == c]` · 위 문제 2 칸의 비율 계산 한 줄

In [ ]:
rates = []
# ★★★ 여기에 세 줄 작성 — 종류마다 비율 하나씩 rates.append(...)

if len(rates) != len(CLASSES):
    print("아직 작성 안 함 — rates 에 열 개가 들어가야 함")
else:
    for c in np.argsort(rates)[::-1]:
        print(f"{which} · {CLASSES[c]:12s} {rates[c]:5.1f}%")

## 문제 3 ★ 기본 — 혼동 행렬을 가로 · 세로로 읽기 (재현율 · 정밀도)

- **왜** — 수업에서 같은 표를 가로로 읽으면 재현율(놓치지 않는가), 세로로 읽으면 정밀도(그 답을 믿을 수 있는가)
- **목표** — 표를 만들고, 종류마다 두 비율을 직접 계산해 막대로 견주기
- **예상 먼저** — 트럭과 자동차 중 '놓치는' 쪽은 어디이고 '잘못 불리는' 쪽은 어디일까
- **◆ 고칠 줄** — `precision` 한 줄의 분모 · 지금은 재현율과 같은 **줄의 합**으로 되어 있음 → **열의 합**으로 고칠 것
- 문법 예시
  - 대각선(맞힌 칸): `cm.diagonal()`
  - 줄마다 합(가로로 더하기): `cm.sum(1)` · 열마다 합(세로로 더하기): `cm.sum(0)`
- 힌트: 표의 줄은 정답 · 칸(열)은 모델의 답

In [ ]:
cm = torch.zeros(len(CLASSES), len(CLASSES), dtype=torch.long)
for t, q in zip(y_val.tolist(), pred.tolist()):
    cm[t, q] += 1                                     # 정답 줄 t · 모델의 답 칸 q 에 1 더하기

diag = cm.diagonal().float()                          # 분자 — 맞힌 칸 (둘 다 같음)
recall = diag / cm.sum(1)                             # 가로 — 정답이 X인 사진 중 X라고 맞힌 비율
precision = diag / cm.sum(1)                          # ◆ 세로 — 모델이 X라고 답한 사진 중 실제로 X인 비율 · 분모를 고칠 것

fig, ax = plt.subplots(1, 2, figsize=(13, 5.2), gridspec_kw={"width_ratios": [1, 1.25]})
ax[0].imshow(cm, cmap="Blues")
ax[0].set_xticks(range(len(CLASSES)), CLASSES, rotation=60, ha="right", fontsize=8)
ax[0].set_yticks(range(len(CLASSES)), CLASSES, fontsize=8)
ax[0].set_xlabel("predicted")
ax[0].set_ylabel("true")
for r in range(len(CLASSES)):
    for c in range(len(CLASSES)):
        ax[0].text(c, r, int(cm[r, c]), ha="center", va="center", fontsize=7,
                   color="white" if cm[r, c] > cm.max() / 2 else "black")
x_ = np.arange(len(CLASSES))
ax[1].bar(x_ - 0.2, recall.numpy(), 0.4, label="recall (row)")
ax[1].bar(x_ + 0.2, precision.numpy(), 0.4, label="precision (column)")
ax[1].set_xticks(x_, CLASSES, rotation=60, ha="right", fontsize=8)
ax[1].set_ylim(0, 1)
ax[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

print(f"{'종류':12s} {'재현율':>6s} {'정밀도':>6s}")
for n, r_, p_ in zip(CLASSES, recall, precision):
    print(f"{n:12s} {r_:6.2f} {p_:6.2f}  {'← 놓침이 더 많음' if r_ < p_ else ''}")

# 확인 — 고치지 않았으면 두 막대가 똑같음
if torch.allclose(recall, precision):
    print("확인 · 두 막대가 모든 종류에서 똑같음 → precision 의 분모가 아직 줄의 합 · 열의 합 cm.sum(0) 으로 고칠 것")
else:
    print(f"확인 · 열의 합 전체 {int(cm.sum(0).sum())} = 사진 장수 · 두 막대가 종류마다 다름 → 제대로 고침")

- 확인 질문 1 — 트럭과 자동차에서 파랑(재현율)과 주황(정밀도) 중 어느 쪽이 낮은가 · 예상과 같은가
- 확인 질문 2 — 틀린 사진 한 장은 두 막대 중 어디에 영향을 주는가
- 흔한 에러
  - `cm.sum(2)` — 표는 2차원이라 방향 번호는 0 · 1 뿐 → `IndexError: Dimension out of range`
  - 정밀도에 `nan` — 모델이 그 종류라고 한 번도 답하지 않아 분모가 0 · 그 종류는 '읽을 수 없음'으로 둠

**막혔을 때 이어갈 코드** — 두 비율을 숫자로만

In [ ]:
cm = torch.zeros(len(CLASSES), len(CLASSES), dtype=torch.long)
for t, q in zip(y_val.tolist(), pred.tolist()):
    cm[t, q] += 1
diag = cm.diagonal().float()
for n, r_, p_ in zip(CLASSES, diag / cm.sum(1), diag / cm.sum(0)):
    print(f"{n:12s} 재현율 {r_:.2f} · 정밀도 {p_:.2f}")

### 문제 3 ★★ 도전 — 한 종류의 줄과 열 열어 보기

- 조건 하나(`name`)를 바꿔, 그 종류를 **놓칠 때 어디로 가는지**(줄) · **잘못 부를 때 무엇이 섞이는지**(열)를 봄

In [ ]:
name = "cat"                                          # ◆ 열어 볼 종류
c = CLASSES.index(name)
row = cm[c].clone(); row[c] = 0                       # 줄 — 정답이 name 인데 다른 답
col = cm[:, c].clone(); col[c] = 0                    # 열 — 답이 name 인데 다른 정답
print(f"{name} 을 놓칠 때 가장 많이 간 답 — {CLASSES[row.argmax()]} {int(row.max())}장 (재현율을 낮춤)")
print(f"{name} 이라고 잘못 부른 사진에 가장 많이 섞인 정답 — {CLASSES[col.argmax()]} {int(col.max())}장 (정밀도를 낮춤)")

### 문제 3 ★★★ 심화 — 확신이 큰 답만 믿으면 정밀도는? (몇 줄 작성)

- 확신 0.9 이상인 사진만 남겨 종류마다 정밀도를 다시 구해 `prec_hi` 에 담을 것 · 남는 사진 수도 함께 볼 것
- 문법 예시: `keep = conf >= 0.9` · `(pred[keep] == c)` 는 답이 c 인 사진 · `(y_val[keep] == c)` 는 정답이 c 인 사진

In [ ]:
prec_hi = None
# ★★★ 여기에 서너 줄 작성 — 마지막에 prec_hi = 종류마다 정밀도 열 개

if prec_hi is None:
    print("아직 작성 안 함")
else:
    print(f"확신 0.9 이상 남은 사진 {int((conf >= 0.9).sum())}/{len(conf)}장")
    for n, p_all, p_hi in zip(CLASSES, precision, prec_hi):
        print(f"{n:12s} 정밀도 전체 {p_all:.2f} → 확신 큰 답만 {p_hi:.2f}")

## 문제 4 ★ 기본 — 헷갈린 짝 하나를 열어 세 칸으로 적기

- **왜** — 숫자로 짝을 찾은 뒤에는 사진을 열어 봐야 원인 후보를 세울 수 있음 (수업의 세 칸과 같은 형식)
- **목표** — 가장 많이 헷갈린 칸 다섯 개 중 하나를 열어 사진을 보고, 아래 세 칸을 적기
- **◆ 고칠 줄** — `rank` 한 줄 (1 · 2 · 3 · 4 · 5)

In [ ]:
off = cm.clone()
off.fill_diagonal_(0)                                 # 맞힌 칸(대각선)은 빼고
ranked = sorted(((int(off[i, j]), i, j) for i in range(len(CLASSES)) for j in range(len(CLASSES)) if i != j), reverse=True)
for r, (n_, i, j) in enumerate(ranked[:5], start=1):
    print(f"{r}번째로 많이 헷갈린 칸 — 정답 {CLASSES[i]} → 모델의 답 {CLASSES[j]} {n_}장")

rank = 1                                              # ◆ 열어 볼 칸 — 1 · 2 · 3 · 4 · 5
n_, ti, pj = ranked[rank - 1]
sel = ((y_val == ti) & (pred == pj)).nonzero().flatten()[:8]
ok = ((y_val == pj) & (pred == pj)).nonzero().flatten()[:4]   # 비교용 — 모델이 맞힌 '답 쪽' 종류 사진
show([x_val[i] for i in sel] + [x_val[i] for i in ok],
     [f"true {CLASSES[ti]}\npred {CLASSES[pj]}" for _ in sel] + [f"(ok) {CLASSES[pj]}" for _ in ok], cols=8)
print(f"윗줄 — 정답 {CLASSES[ti]} 인데 {CLASSES[pj]} 로 답한 사진 · 아랫줄 (ok) — 모델이 맞힌 {CLASSES[pj]} 사진")

아래 칸을 두 번 눌러 고칠 것. 방금 연 짝 하나를 적음 · 공통점이 뚜렷하지 않으면 없다고 적어도 됨.

- **고른 짝** —
- **보이는 사실** (사진에서 짚을 수 있는 차이) —
- **원인 후보** —
- **확인할 비교** (하나만 바꾸고 나머지는 그대로) —

- 확인 질문 — 내가 적은 '확인할 비교'에서 바꾸는 것은 하나뿐인가 · 그대로 두는 것은 무엇인가
- 흔한 에러
  - `IndexError: list index out of range` — `rank` 가 0 이하이거나 너무 큼 · 1부터 셈
  - `NameError: name 'cm' is not defined` — 문제 3 칸을 건너뜀 · 문제 3 칸이나 그 아래 '이어갈 코드'를 먼저 실행

### 문제 4 ★★ 도전 — 둘째 짝도 같은 설명이 맞는가

- `rank` 를 2 로 바꿔 한 번 더 실행하고, 첫 짝에서 적은 '보이는 사실'이 둘째 짝에도 맞는지 한 줄 적기 · 맞지 않으면 원인 후보가 짝마다 다르다는 뜻

### 문제 4 ★★★ 심화 — '확인할 비교' 하나를 코드로 (몇 줄 작성)

- 연 짝의 사진(`sel` 전체)을 좌우 반전했을 때 정답으로 돌아오는 장수를 셀 것 · 반전이 원인이 아니라면 거의 돌아오지 않아야 함
- 문법 예시: `sel_all = ((y_val == ti) & (pred == pj)).nonzero().flatten()` · `predict(model, x_val[sel_all].float().flip(-1)).argmax(1)`

In [ ]:
back = None
# ★★★ 여기에 세 줄 작성 — 마지막에 back = 정답으로 돌아온 장수

if back is None:
    print("아직 작성 안 함")
else:
    n_all = int(((y_val == ti) & (pred == pj)).sum())
    print(f"정답 {CLASSES[ti]} → 답 {CLASSES[pj]} 사진 {n_all}장 중 좌우 반전 뒤 정답으로 돌아온 사진 {back}장")